In [1]:
import os
import pandas as pd
from parser.utils import parse_cfr_xml, flatten_volume
from parser.types import Volume, Content
from typing import List

In [2]:
all_volumes = []

for f in os.walk("title-12/"):
    root, _, files = f
    for name in files:
      all_volumes.append(os.path.join(root, name))

all_volumes

['title-12/CFR-2024-title12-vol9.xml',
 'title-12/CFR-2024-title12-vol4.xml',
 'title-12/CFR-2024-title12-vol5.xml',
 'title-12/CFR-2024-title12-vol3.xml',
 'title-12/CFR-2024-title12-vol7.xml',
 'title-12/CFR-2024-title12-vol8.xml',
 'title-12/CFR-2024-title12-vol1.xml',
 'title-12/CFR-2024-title12-vol10.xml',
 'title-12/CFR-2024-title12-vol2.xml']

In [3]:
def load_volumes(vol_list):
  for vol_path in vol_list:
    parsed = parse_cfr_xml(vol_path)
    yield (vol_path, flatten_volume(Volume.model_validate(parsed)))

volumes = load_volumes(all_volumes)

In [4]:
def extract_text_from_content(contents: List[Content]) -> str:
    texts = []
    for content in contents:
        if content.text:
            texts.append(content.text.strip())
        if content.subparagraphs:
            texts.extend([sub.text.strip() for sub in content.subparagraphs if sub.text])
    return "\n\n".join(texts)

def chunk_volume(volume: Volume):
    chunks = []
    for part in volume.parts:
        all_sections = part.sections.copy()
        for subpart in part.subparts:
            all_sections.extend(subpart.sections)

        for section in all_sections:
            if not section.content:
                continue

            chunk = {
                "volume": volume.metadata.title_number,
                "part": part.number,
                "section": section.number,
                "title": section.subject,
                "text": extract_text_from_content(section.content)
            }
            chunks.append(chunk)

    return chunks

In [4]:
# parsed = parse_cfr_xml("title-12/CFR-2024-title12-vol1.xml")
# volume = Volume.model_validate(parsed)

df_dict = {}

for path, volume in volumes:
  path = path.replace(".xml", "").split("-")[-1]
  df_dict[path] = volume
  volume.to_csv(f"{path}.csv")

In [5]:
df_dict["vol2"].head()

,title_number,metadata_subject,metadata_parts,metadata_revised,metadata_contains,metadata_date,metadata_publication,part_number,subpart_title,section_number,section_subject,content_type,content_heading,content_text
0,Title 12,Banks and Banking,Parts 200 to 219,"Revised as of January 1, 2024",Containing a codification of documents of gene...,"As of January 1, 2024",Published by the Office of the Federal Registe...,None,None,§ 201.1,"Authority, purpose and scope.",paragraph,(a),Authority. This part is issued under the autho...
1,Title 12,Banks and Banking,Parts 200 to 219,"Revised as of January 1, 2024",Containing a codification of documents of gene...,"As of January 1, 2024",Published by the Office of the Federal Registe...,None,None,§ 201.1,"Authority, purpose and scope.",paragraph,(b),Purpose and scope. This part establishes rules...
2,Title 12,Banks and Banking,Parts 200 to 219,"Revised as of January 1, 2024",Containing a codification of documents of gene...,"As of January 1, 2024",Published by the Office of the Federal Registe...,None,None,§ 201.2,Definitions.,paragraph,None,"For purposes of this part, the following defin..."
3,Title 12,Banks and Banking,Parts 200 to 219,"Revised as of January 1, 2024",Containing a codification of documents of gene...,"As of January 1, 2024",Published by the Office of the Federal Registe...,None,None,§ 201.2,Definitions.,paragraph,(a),Appropriate federal banking agency has the sam...
4,Title 12,Banks and Banking,Parts 200 to 219,"Revised as of January 1, 2024",Containing a codification of documents of gene...,"As of January 1, 2024",Published by the Office of the Federal Registe...,None,None,§ 201.2,Definitions.,paragraph,(b),Critically undercapitalized insured depository...
